# Variadic voltage controlled DC/DC converter schema

## Schema
<div style="text-align: center;">
<img src="./principle_diagram.drawio.svg" alt="Principle diagram" style="display: block; margin: 0 auto; width: 50%;"/>
</div>

### Initial Conditions

In [10]:
from IPython.display import display, Markdown

MinVout = 5  # V, the main output voltage
MaxVout = 15  # V, the secondary output voltage (switchable between 9V and 12V)
Vref = 1.25  # V, the reference voltage of the XL6019 regulator
R9 = 1000  # Ohms, the resistance of R1
Rds = 5  # Ohms, the on-resistance of the MOSFET
Fs = 10000  # Hz, the switching frequency of the regulator
V_pwm = 3.3  # V, the voltage of the PWM signal
n = 3  # the number of RC-filter stages

display(
    Markdown(
        f"__The minimum output voltage is $V_{{out, min}} = {MinVout}\\,\\text{{V}}$ and the maximum output voltage is $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)
display(
    Markdown(
        f"__The reference voltage of the XL6019 regulator is $V_{{ref}} = {Vref}\\,\\text{{V}}$.__"
    )
)
display(Markdown(f"__The resistance of R9 is $R_{{9}} = {R9}\\,\\text{{Ohms}}$.__"))
display(
    Markdown(
        f"__The on-resistance of the MOSFET is $R_{{ds}} = {Rds}\\,\\text{{Ohms}}$.__"
    )
)
display(
    Markdown(
        f"__The switching frequency of the regulator is $F_s = {Fs / 1000}\\,\\text{{kHz}}$.__"
    )
)

__The minimum output voltage is $V_{out, min} = 5\,\text{V}$ and the maximum output voltage is $V_{out, max} = 15\,\text{V}$.__

__The reference voltage of the XL6019 regulator is $V_{ref} = 1.25\,\text{V}$.__

__The resistance of R9 is $R_{9} = 1000\,\text{Ohms}$.__

__The on-resistance of the MOSFET is $R_{ds} = 5\,\text{Ohms}$.__

__The switching frequency of the regulator is $F_s = 10.0\,\text{kHz}$.__

The DC/DC converter voltage formula:

$$
V_{out} = V_{ref} \cdot \left( 1 + \frac {R_{top}}{R_{bottom}}\right)
$$

, where the $R_{top}$ - resistor connected to $V_{out}$ terminal, the  $R_{bottom}$ - resistor connected to ground, the $V_{ref}$ - reference voltage of DC/DC internal comparator.

For the minimum voltage $V_{out,min}$ the equivalent schema of divider $R_{top}$, $R_{bottom}$ would look like:

<div style="text-align: center;">
<img src="./equivalent_diagram_fb_divider.drawio.svg" alt="Principle diagram" style="display: block; margin: 0 auto; width: 10%;"/>
</div>

Hence the $R_9$ is provided, the $R_7$ can be derived from formula above:

$$
V_{out,min} = V_{ref} \cdot \left( 1 + \frac {R_{7}}{R_{9}}\right) \implies R_{7} = R_{9}\left(\frac {V_{out,min}}{V_{ref}} -1\right) \text{, where: } \frac {V_{out,min}}{V_{ref}} = G_{dc/dc} 
$$

$$
\frac {V_{out,min}}{V_{ref}} = G_{dc/dc,min} \text{ - gain of DC/DC converter for minimum output voltage;}
$$

$$
G_{dc/dc,min} -1 = K_{r, min}\text{ - resistor divider factor for minimum output voltage.}
$$

Therefore:

$$
R_{7} = R_{9} \cdot K_{r, min}
$$

In [11]:
Gdcdc_min = MinVout / Vref
Kr_min = Gdcdc_min - 1
R7 = R9 * Kr_min

display(
    Markdown(
        f"__The minimum gain of the DC-DC converter is $G_{{dcdc, min}} = {Gdcdc_min:.2f}$ and the minimum gain ratio is $K_{{r, min}} = {Kr_min:.2f}$.__"
    )
)
display(
    Markdown(
        f"__The resistance of R7 is $R_{{7}} = {R7:.2f}\\,\\text{{Ohms}}$ for the case of the minimum output voltage $V_{{out, min}} = {MinVout}\\,\\text{{V}}$.__"
    )
)

__The minimum gain of the DC-DC converter is $G_{dcdc, min} = 4.00$ and the minimum gain ratio is $K_{r, min} = 3.00$.__

__The resistance of R7 is $R_{7} = 3000.00\,\text{Ohms}$ for the case of the minimum output voltage $V_{out, min} = 5\,\text{V}$.__

In order to determine the DC/DC feedback divider configuration for maximum voltage $V_{out,max}$ let's consider the following equivalent schema:

<div style="text-align: center;">
<img src="./equivalent_diagram_fb_divider_for_v_max.drawio.svg" alt="Equivalent divider schema for maximum voltage" style="display: block; margin: 0 auto; width: 50%;"/>
</div>

The $V_{out,max}$ for this schema is equal:

$$
V_{out,max} = V_{ref} \cdot \left( 1 + \frac {R_{7}}{R_{eq}}\right) \implies R_{eq} = \frac {R_{7}}{K_{r,max}} \text{, where: }
$$ 

$$
\frac {V_{out,max}}{V_{ref}} = G_{dc/dc,max} \text{ - gain of DC/DC converter for maximum output voltage;}
$$

$$
G_{dc/dc,max} -1 = K_{r, max}\text{ - resistor divider factor for maximum output voltage.}
$$

In [12]:
Gdcdc_max = MaxVout / Vref
Kr_max = Gdcdc_max - 1
Req = R7 / Kr_max
display(
    Markdown(
        f"__The maximum gain of the DC-DC converter is $G_{{dcdc, max}} = {Gdcdc_max:.2f}$ and the maximum gain ratio is $K_{{r, max}} = {Kr_max:.2f}$.__"
    )
)
display(
    Markdown(
        f"__The equivalent resistance $R_{{eq}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$ is $R_{{eq}} = {Req:.2f}\\,\\text{{Ohms}}$.__"
    )
)

__The maximum gain of the DC-DC converter is $G_{dcdc, max} = 12.00$ and the maximum gain ratio is $K_{r, max} = 11.00$.__

__The equivalent resistance $R_{eq}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$ is $R_{eq} = 272.73\,\text{Ohms}$.__

Let's derive the $R_{8eq}$:

$$
\frac {1}{R_{eq}} = \frac{1}{R_{8eq}+R_{ds}} + \frac {1}{R_9} \implies R_{8eq} = \frac{1}{\frac {1}{R_{eq}} - \frac {1}{R_{9}}} - R_{ds}
$$

In [13]:
g_eq = 1 / Req
g_9 = 1 / R9

R8eq = 1 / (g_eq - g_9) - Rds

display(
    Markdown(
        f"__The equivalent resistance $R_{{8, eq}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$ is $R_{{8, eq}} = {R8eq:.2f}\\,\\text{{Ohms}}$.__"
    )
)

__The equivalent resistance $R_{8, eq}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$ is $R_{8, eq} = 370.00\,\text{Ohms}$.__

And gain $G_{Imin}$ of the divider above is:

<div style="text-align: center;">
<img src="./r8eq_rds_divider_gain.drawio.svg" alt="Divider gain schema" style="display: block; margin: 0 auto; width: 20%;"/>
</div>

$$
G_{Vref} = \frac {R_{ds}}{R_{8eq}} + 1
$$

In [14]:
GVref = Rds / R8eq + 1

display(
    Markdown(
        f"__The reference voltage gain is $G_{{V, ref}} = {GVref:.2f}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)

__The reference voltage gain is $G_{V, ref} = 1.01$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__

In case of maximum output voltage $V_{out,max}$ the PWM signal transfer function will look like:

<div style="text-align: center;">
<img src="./transfer_function_diagram.drawio.svg" alt="PWM signal transfer function" style="display: block; margin: 0 auto; width: 40%;"/>
</div>

$$
V_{pwm} \cdot G_{fl1,dc} \cdot G_{V_{Imax},dc} \cdot G_{Vref} = V_{ref} \implies G_{fl1,dc} \cdot G_{V_{Imax},dc} \cdot G_{Vref} = \frac {V_{ref}}{V_{pwm}} 
$$

The $G_{V_{Imax},dc}$ for non-inverting operation amplifier is:

$$
G_{V_{Imax},dc} = \frac {R_6 + R_5}{R_5} = \frac {R_6}{R_5} + 1
$$

Let's pick the overall gain of last two cascades $G_{V_{Imax},dc}$ and $G_{Vref}$ equal 2:

$$
G_{cc} = G_{V_{Imax},dc} \cdot G_{Vref} = 2 \implies G_{V_{Imax},dc} = \frac {G_{cc}}{G_{Vref}}= \frac {2}{G_{Vref}}
$$

$$
G_{fl1,dc} \cdot G_{V_{Imax},dc} \cdot G_{Vref} = \frac {V_{ref}}{V_{pwm}} \implies G_{fl1,dc} \cdot G_{cc} = \frac {V_{ref}}{V_{pwm}} \implies
$$

$$
\implies G_{fl1,dc} = \frac {V_{ref}}{V_{pwm} \cdot G_{cc}}
$$

In [15]:
Gcc = 2
GVImax_dc = Gcc / GVref

Gfl1_dc = Vref / (V_pwm * Gcc)

display(
    Markdown(
        f"__The maximum voltage gain of the error amplifier in DC conditions is $G_{{VI, max, dc}} = {GVImax_dc:.2f}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)
display(
    Markdown(
        f"__The required gain of the PWM frequency filter cascade by DC is $G_{{fl1, dc}} = {Gfl1_dc:.2f}$.__"
    )
)

__The maximum voltage gain of the error amplifier in DC conditions is $G_{VI, max, dc} = 1.97$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__

__The required gain of the PWM frequency filter cascade by DC is $G_{fl1, dc} = 0.19$.__

In order to calculate the $R_5$, $R_6$ and $R_8$ let's pick some constraints:

- let's N = 10 - number of times the $R_8$ resistor is less than $(R_6+R_5)$;

$$
\left\{
\begin{array}{rcl}
    \frac{1}{N} \cdot (R_6+R_5) &=& R_8 \\
    R_8 ||(R_6+R_5) &=& R_{8eq}
\end{array}
\right. \text{, let } R=(R_6+R_5) \implies 
\left\{
\begin{array}{rcl}
    \frac{1}{N} \cdot R &=& R_8 \\
    \frac {1} {R_8} + \frac {1} {R} &=& \frac {1} {R_{8eq}}
\end{array}
\right. = 
$$

$$
= \left\{
\begin{array}{rcl}
    R_8 &=& \frac{1}{N} \cdot R \\
    \frac {1} {\frac{1}{N} \cdot R} + \frac {1} {R} &=& \frac {1} {R_{8eq}}
\end{array}
\right. = 
\left\{
\begin{array}{rcl}
    R_8 &=& \frac{1}{N} \cdot R \\
    \frac {R+ \frac {1}{N}R} {\frac{1}{N} \cdot R^2} &=& \frac {1} {R_{8eq}}
\end{array}
\right. = 
\left\{
\begin{array}{rcl}
    R_8 &=& (1 + \frac{1}{N}) R_{8eq} \\
    R &=& (N+1)R_{8eq}
\end{array}
\right.
$$

Next, to find $R_5$ and $R_6$ resistors values, we need to solve the following system of equations:

$$
\left\{
\begin{array}{rcl}
    \frac{R_6}{R_5} &=& G_{V_{Imax},dc} \\
    R_6+R_5 &=& R
\end{array}
\right. = 
\left\{
\begin{array}{rcl}
    R_6 &=& \left(G_{V_{Imax},dc} - 1\right)R_5\\
    R &=& \left(G_{V_{Imax},dc} - 1\right)R_5+R_5
\end{array}
\right. = 
\left\{
\begin{array}{rcl}
    R_6 &=& R\left(1- \frac {1}{G_{V_{Imax},dc}}\right)\\
    R_5 &=& \frac {R}{G_{V_{Imax},dc}}
\end{array}
\right.
$$

In [16]:
N = 20
R8 = R8eq * (1 + 1 / N)
R = R8eq * (N + 1)

R6 = R * (1 - 1 / (GVImax_dc))
R5 = R / GVImax_dc

display(
    Markdown(
        f"__The resistance of R8 is $R_{{8}} = {R8:.2f}\\,\\text{{Ohms}}$, the resistance of R6 is $R_{{6}} = {R6:.2f}\\,\\text{{Ohms}}$, and the resistance of R5 is $R_{{5}} = {R5:.2f}\\,\\text{{Ohms}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)

__The resistance of R8 is $R_{8} = 388.50\,\text{Ohms}$, the resistance of R6 is $R_{6} = 3832.50\,\text{Ohms}$, and the resistance of R5 is $R_{5} = 3937.50\,\text{Ohms}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__

### PWM RC-filter

<div style="text-align: center;">
<img src="./rc_filter.drawio.svg" alt="Equivalent RC-filter circuit" style="display: block; margin: 0 auto; width: 30%;"/>
</div>

To determine resistor values we must perform calculation for DC case. So let's consider the equivalent schema of the filter for DC:

<div style="text-align: center;">
<img src="./rc_filter_dc_equivalent.drawio.svg" alt="Equivalent RC-filter circuit" style="display: block; margin: 0 auto; width: 50%;"/>
</div>


$$
R_{eq,fl1} = R_1 + R_2 + R_3...+ R_n\text{, } n \text{ - number of RC-cascades}
$$


Let's determine the DC gain of the divider:

$$
G_{fl1, dc} = \frac{R_4}{R_{eq,fl1} + R_4}
$$

So, at this point we have got two unknown variables: $R_{eq,fl1}$ and $R_4$. To solve the task we need introduce some constraint. Luckily we have OpAmp cascade connected to the non-inverting input of the RC-filter/divider. 

<div style="text-align: center;">
<img src="./rc_filter_opamp_i_bias.drawio.svg" alt="Equivalent RC-filter circuit" style="display: block; margin: 0 auto; width: 25%;"/>
</div>

So, as any OpAmp, this one is not ideal - it has some bias current $I_b$ applied to both its terminals. This bias current must be compensated with some $R_b$ connected in series to positive terminal:

<div style="text-align: center;">
<img src="./opamp_i_bias.drawio.svg" alt="Equivalent RC-filter circuit" style="display: block; margin: 0 auto; width: 25%;"/>
</div>

The formula of the bias correcting resistor $R_b$ is:

$$
R_b = R_1 || R_2 = \frac {R_1 \cdot R_2}{R_1 + R_2}
$$

For our case:

<div style="text-align: center;">
<img src="./opamp_i_bias_with_rc_as_resistance.drawio.svg" alt="Equivalent RC-filter circuit" style="display: block; margin: 0 auto; width: 50%;"/>
</div>

$$
R_b = R_{f} = R_5 || R_6 = \frac {R_5 \cdot R_6}{R_5 + R_6}\text{, where: }  R_{f} = R_{eq,fl1} || R_4 = \frac {R_{eq,fl1} \cdot R_4}{R_{eq,fl1} + R_4}
$$

Now, let's put all the constraints to the system and solve it:

$$
\left\{
\begin{array}{rcl}
    \frac{R_4}{R_{eq,fl1} + R_4} &=& G_{fl1, dc} \\
    \frac {R_{eq,fl1} \cdot R_4}{R_{eq,fl1} + R_4} &=& R_{f}
\end{array}
\right. = 
\left\{
\begin{array}{rcl}
    R_{eq,fl1} &=& \frac {\frac{1}{G_{fl1, dc}} - 1}{ 1 - G_{fl1, dc} } \cdot R_{f} \\
    R_4 &=& \frac {R_{f}}{ 1 - G_{fl1, dc} }
\end{array}
\right.
$$

In [17]:
Rb = R5 * R6 / (R5 + R6)
Rf = Rb

Greverse = 1 / Gfl1_dc

Req_lf1 = (Greverse - 1) * Rf / (1 - Gfl1_dc)
R4 = Rf / (1 - Gfl1_dc)

R1 = Req_lf1 / n
R2 = R1
R3 = R1

display(
    Markdown(
        f"__The resistance of R4 is $R_{{4}} = {R4:.2f}\\,\\text{{Ohms}}$ and the equivalent resistance $R_ {{eq, lf1}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$ is $R_{{eq, lf1}} = {Req_lf1:.2f}\\,\\text{{Ohms}}$.__"
    )
)

display(
    Markdown(
        f"__The resistance of R1, R2, and R3 is $R_{{1}} = R_{{2}} = R_{{3}} = {R1:.2f}\\,\\text{{Ohms}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)

__The resistance of R4 is $R_{4} = 2395.92\,\text{Ohms}$ and the equivalent resistance $R_ {eq, lf1}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$ is $R_{eq, lf1} = 10254.53\,\text{Ohms}$.__

__The resistance of R1, R2, and R3 is $R_{1} = R_{2} = R_{3} = 3418.18\,\text{Ohms}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__

## Result

In [18]:
display(
    Markdown(
        f"__The voltage of the PWM signal is $V_{{PWM}} = {V_pwm}\\,\\text{{V}}$.__"
    )
)
display(Markdown(f"__The R9 is $R_{{9}} = {R9:.2f}\\,\\text{{Ohms}}$.__"))
display(
    Markdown(
        f"__The resistance of R7 is $R_{{7}} = {R7:.2f}\\,\\text{{Ohms}}$ for the case of the minimum output voltage $V_{{out, min}} = {MinVout}\\,\\text{{V}}$.__"
    )
)
display(
    Markdown(
        f"__The resistance of R8 is $R_{{8}} = {R8:.2f}\\,\\text{{Ohms}}$, the resistance of R6 is $R_{{6}} = {R6:.2f}\\,\\text{{Ohms}}$, and the resistance of R5 is $R_{{5}} = {R5:.2f}\\,\\text{{Ohms}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)
display(
    Markdown(
        f"__The resistance of R4 is $R_{{4}} = {R4:.2f}\\,\\text{{Ohms}}$ and the equivalent resistance $R_ {{eq, lf1}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$ is $R_{{eq, lf1}} = {Req_lf1:.2f}\\,\\text{{Ohms}}$.__"
    )
)

display(
    Markdown(
        f"__The resistance of R1, R2, and R3 is $R_{{1}} = R_{{2}} = R_{{3}} = {R1:.2f}\\,\\text{{Ohms}}$ for the case of the maximum output voltage $V_{{out, max}} = {MaxVout}\\,\\text{{V}}$.__"
    )
)

__The voltage of the PWM signal is $V_{PWM} = 3.3\,\text{V}$.__

__The R9 is $R_{9} = 1000.00\,\text{Ohms}$.__

__The resistance of R7 is $R_{7} = 3000.00\,\text{Ohms}$ for the case of the minimum output voltage $V_{out, min} = 5\,\text{V}$.__

__The resistance of R8 is $R_{8} = 388.50\,\text{Ohms}$, the resistance of R6 is $R_{6} = 3832.50\,\text{Ohms}$, and the resistance of R5 is $R_{5} = 3937.50\,\text{Ohms}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__

__The resistance of R4 is $R_{4} = 2395.92\,\text{Ohms}$ and the equivalent resistance $R_ {eq, lf1}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$ is $R_{eq, lf1} = 10254.53\,\text{Ohms}$.__

__The resistance of R1, R2, and R3 is $R_{1} = R_{2} = R_{3} = 3418.18\,\text{Ohms}$ for the case of the maximum output voltage $V_{out, max} = 15\,\text{V}$.__